# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all record sets and their fields by `@id`
print("Available Record Sets:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}, name: {rs.name}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      Field @id: {field.id}, name: {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets are referenced by their `@id`. Fields in these sets can be referenced by their own `@id`s.

In [ ]:
# Let's extract data from all record sets found
import pprint

# Gather all record set IDs
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for rs_id in record_set_ids:
    # This loads ALL fields in the record set. For large datasets, consider sampling or specific fields.
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded record set: {rs_id}, {len(df)} records, columns: {df.columns.tolist()}")

# For demonstration, pick the first record set to view its data
first_record_set_id = record_set_ids[0] if record_set_ids else None
if first_record_set_id:
    print(f"Previewing first 5 rows from record set: {first_record_set_id}")
    display(dataframes[first_record_set_id].head())
else:
    print("No record sets found in the metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering, normalization, grouping, or visualization on the dataset. For demonstration, we use GAD-7 scores if present (commonly used in mental health screening). All column references use their field `@id`.

In [ ]:
import numpy as np
import warnings
# Suppress SettingWithCopyWarning for demonstration purposes
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Identify a record set containing GAD-7 scores (by column name or field @id)
gad7_field_id = None
gad7_record_set_id = None
for rs in record_sets:
    for field in rs.fields:
        # We'll match field name (case-insensitive contains 'gad' or 'gad-7')
        if 'gad' in field.name.lower() or 'gad-7' in field.name.lower():
            gad7_field_id = field.id
            gad7_record_set_id = rs.id
            print(f"Found GAD-7 field: {gad7_field_id} in record set: {gad7_record_set_id}")
            break
    if gad7_field_id:
        break

if gad7_record_set_id and gad7_field_id:
    df = dataframes[gad7_record_set_id]
    # Check field present in columns (as @id)
    if gad7_field_id in df.columns:
        # Remove missing/NaN values
        gad7_values = pd.to_numeric(df[gad7_field_id], errors='coerce')
        df = df.assign(gad7_score=gad7_values)
        threshold = 10
        filtered_df = df[df.gad7_score > threshold]
        print(f"Filtered records with {gad7_field_id} (GAD-7) > {threshold}:")
        display(filtered_df.head())

        # Normalize GAD-7 scores
        mean = gad7_values.mean()
        std = gad7_values.std()
        df['gad7_score_normalized'] = (gad7_values - mean) / std
        print(f"Normalized {gad7_field_id} (GAD-7) scores:")
        display(df[[gad7_field_id, 'gad7_score', 'gad7_score_normalized']].head())

        # Grouping example: by gender or another demographic if available
        group_field_id = None
        possible_group_fields = ['gender', 'sex', 'marital_status']
        found_group_field = False
        for field in dataset.metadata.field_list:
            if any(x in field.name.lower() for x in possible_group_fields):
                group_field_id = field.id
                if group_field_id in df.columns:
                    found_group_field = True
                    print(f"Grouping by field: {group_field_id} ({field.name})")
                    grouped_df = df.groupby(group_field_id)[['gad7_score', 'gad7_score_normalized']].mean().reset_index()
                    display(grouped_df)
                    break
        if not found_group_field:
            print("No demographic group field found for grouping in this record set.")
    else:
        print(f"Field {gad7_field_id} not found in DataFrame columns.")
else:
    print("GAD-7 field not found in dataset fields.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we plot the GAD-7 score distribution if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize GAD-7 score distribution if previously identified
if gad7_field_id and gad7_record_set_id and gad7_field_id in dataframes[gad7_record_set_id].columns:
    df = dataframes[gad7_record_set_id]
    gad7_scores = pd.to_numeric(df[gad7_field_id], errors='coerce').dropna()
    plt.figure(figsize=(8, 5))
    sns.histplot(gad7_scores, bins=10, kde=True)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()
else:
    print("GAD-7 score column not available for visualization.")

## 6. Conclusion
This notebook showcased exploratory analysis of the FAIR² Mental Health Survey Dataset using `mlcroissant`.

- Data was loaded and inspected using semantic references via `@id` fields.
- GAD-7 mental health scores and other fields (if present) were explored and visualized.
- The notebook demonstrates how to take advantage of Croissant metadata for reproducible and robust data science workflows in health survey data.

Feel free to extend the analysis to other fields such as PHQ-9 or PSQ, or demographic variables, by referencing their `@id` as seen above.